# 03. 上下文管理与压缩

AGDR VLA 入门教程 · 第 3 章 · 前半周

- **学习目标**：理解 Agent 的上下文为什么会越来越长，掌握截断、清除旧工具结果、摘要压缩三种处理方法及各自的代价，并给 mini agent 加上自动压缩
- **前置知识**：[02-手写AgentLoop](./02-手写AgentLoop.ipynb)
- **预计学习时长**：1.5 小时

## 问题从哪里来

回顾前两节的两个事实：

1. **模型是无状态的**，每次调用都要发送完整的消息历史
2. **Agent 每调用一次工具，历史就多两条消息**，其中工具结果往往很长：一个源代码文件、一段训练日志、一次测试输出

于是一个运行了几十步的 Agent 会遇到三个问题：

| 问题 | 表现 |
| --- | --- |
| 贵 | 每一步都要为全部历史付费，前面读过的大文件之后每一步都在重复计费 |
| 慢 | 输入越长，模型处理得越慢 |
| 超限 | 超过模型的上下文窗口，请求直接报错 |

还有一个不那么明显的问题：上下文很长时，模型**不一定能用好**其中的每一条信息。真正重要的内容被埋在大量无关的工具输出里，模型更容易遗漏或混淆。所以上下文不是「装得下就行」，而是**越精简越好**。

**上下文管理**要回答的就是：上下文快满的时候，留下什么、丢掉什么。

## 准备工作

02 节写的 mini agent 已经整理成了同目录下的 [mini_agent.py](./mini_agent.py)，内容和 02 节相同。本节直接导入它，不再重复那些代码。

> 如果你修改了 `mini_agent.py`，需要重启 notebook 的内核（Kernel → Restart），修改才会生效。

In [ ]:
import copy
import json
import re

import mini_agent as ma

print("模式:", "在线" if ma.ONLINE else "离线（读取录制的模型响应）")
print("可用工具:", list(ma.TOOLS))

### 估算 token 数

要管理上下文，先得知道它有多长。准确的 token 数只有模型的分词器知道，API 响应里的 `usage` 会告诉你**上一次**用了多少。但 Agent 需要在**发送之前**就判断要不要压缩，所以通常会先粗略估算。

下面的估算规则很粗糙：一个汉字算 1 个 token，其他字符每 4 个算 1 个 token。不同模型的分词器差别很大，它只用来判断「大概多长」，不能用来算钱。

In [ ]:
def estimate_tokens(messages):
    text = json.dumps(messages, ensure_ascii=False)
    cjk = len(re.findall(r"[一-鿿]", text))  # 汉字的个数
    return cjk + (len(text) - cjk) // 4


print(estimate_tokens([{"role": "user", "content": "什么是 VLA 模型？"}]))
print(estimate_tokens([{"role": "user", "content": "What is a VLA model?"}]))

## 构造一段很长的历史

为了看清问题，我们手工构造一段「Agent 已经工作了一阵子」的消息历史：用户问为什么这周的训练比上周慢，Agent 列了目录，读了两份各 120 行的训练日志，又读了配置文件。

这些消息的结构和 02 节运行时产生的完全一样，只是内容是我们写好的。

In [ ]:
def fake_log(num_workers, data_time, steps=120):
    # 生成一份模拟训练日志：开头一行配置，之后每步一行
    lines = [f"[INFO] config num_workers={num_workers} batch_size=32"]
    for step in range(1, steps + 1):
        loss = 2.5 * 0.985 ** step
        dt = data_time + 0.01 * (step % 3)
        lines.append(f"[INFO] step={step} loss={loss:.3f} data_time={dt:.2f} compute_time=0.31")
    return "\n".join(lines)


def tool_round(call_id, name, args, result, text=None):
    # 一轮工具调用 = 一条 assistant（调用请求）+ 一条 tool（结果）
    call = {"id": call_id, "type": "function",
            "function": {"name": name, "arguments": json.dumps(args, ensure_ascii=False)}}
    return [
        {"role": "assistant", "content": text, "tool_calls": [call]},
        {"role": "tool", "tool_call_id": call_id, "content": result},
    ]


long_history = [
    {"role": "system", "content": ma.SYSTEM_PROMPT},
    {"role": "user", "content": "这周的训练比上周慢了很多，帮我查一下原因。"},
    *tool_round("call_1", "list_dir", {"path": "."},
                "config.json\ntrain_last_week.log\ntrain_this_week.log", "先看看有哪些文件。"),
    *tool_round("call_2", "read_file", {"path": "train_this_week.log"}, fake_log(2, 0.44)),
    *tool_round("call_3", "read_file", {"path": "train_last_week.log"}, fake_log(8, 0.11)),
    *tool_round("call_4", "read_file", {"path": "config.json"},
                '{"lr": 0.0003, "batch_size": 32, "num_workers": 2}'),
]

print("消息条数:", len(long_history))
print("估算 token:", estimate_tokens(long_history))
tool_tokens = estimate_tokens([m for m in long_history if m["role"] == "tool"])
print(f"其中工具结果约占 {tool_tokens / estimate_tokens(long_history):.0%}")

绝大部分 token 都花在了工具结果上，而真正有用的信息只有几行：两份日志里的 `num_workers` 和 `data_time`。这是 Agent 上下文的典型状况。

下面依次尝试三种处理方法。

## 方法一：截断，只保留最近的消息

最简单的想法：只保留系统提示词和最后几条消息。我们先「天真」地截一刀，再用一个小工具检查结果是否合法。

`check_history` 检查一条规则：**每一条 `tool` 消息，前面都必须有一条 `assistant` 消息发起了对应 `tool_call_id` 的调用。** 违反这条规则的请求会被 API 拒绝。

In [ ]:
def check_history(messages):
    pending = set()  # 已经发起、还没收到结果的调用编号
    for i, m in enumerate(messages):
        for call in m.get("tool_calls") or []:
            pending.add(call["id"])
        if m["role"] == "tool":
            if m["tool_call_id"] not in pending:
                return f"❌ 第 {i} 条是工具结果，但找不到发起它的调用 {m['tool_call_id']}"
            pending.discard(m["tool_call_id"])
    roles = [m["role"] for m in messages]
    if "user" not in roles:
        return "⚠️ 格式合法，但已经没有任何 user 消息了：模型不知道任务是什么"
    return "✅ 合法"


naive = [long_history[0]] + long_history[-3:]
for m in naive:
    print(m["role"], "|", (m.get("content") or "调用 " + m["tool_calls"][0]["function"]["name"])[:40])
print(check_history(naive))

天真的截断犯了两个错误：

1. 从一轮工具调用的**中间**截断，留下了一条「孤儿」工具结果，API 会直接报错
2. 把用户的**原始任务**也截掉了。即使格式合法，模型也不知道自己在做什么

正确的截断要按「轮」来切：始终保留系统提示词和原始任务，然后从某一轮的 `assistant` 消息开始保留。

In [ ]:
def trim_history(messages, keep_rounds=1):
    head = messages[:2]  # 系统提示词 + 原始任务，永远保留
    round_starts = [i for i, m in enumerate(messages) if i >= 2 and m["role"] == "assistant"]
    if len(round_starts) <= keep_rounds:
        return list(messages)
    return head + messages[round_starts[-keep_rounds]:]


trimmed = trim_history(long_history, keep_rounds=1)
print(check_history(trimmed))
print("消息条数:", len(long_history), "→", len(trimmed))
print("估算 token:", estimate_tokens(long_history), "→", estimate_tokens(trimmed))

格式合法了，体积也大幅缩小。但代价很明显：模型**完全忘了**自己读过两份日志。它只看到 config.json 里写着 `num_workers: 2`，不知道上周是 8。它可能重新去读日志（又花一遍 token），也可能直接给出错误的结论。

截断适合「旧内容确实没用了」的场景，例如闲聊式的长对话。对需要前后串联信息的 Agent 任务，它太粗暴了。

## 方法二：清除旧的工具结果

观察到「占地方的主要是工具结果」，可以只动工具结果：**保留所有消息的结构**，把较早的工具结果替换成一句占位说明。

这样模型仍然知道自己做过哪些操作（调用请求还在），只是看不到旧结果的具体内容；需要时可以重新调用工具。这是一种很常用、也很「便宜」的做法，不需要额外调用模型。

In [ ]:
PLACEHOLDER = "[这个工具结果已被清除以节省上下文。如果还需要其中的内容，请重新调用工具。]"


def clear_old_tool_results(messages, keep_last=1):
    result = copy.deepcopy(messages)  # 不修改原来的历史
    tool_indices = [i for i, m in enumerate(result) if m["role"] == "tool"]
    for i in tool_indices[:-keep_last] if keep_last else tool_indices:
        result[i]["content"] = PLACEHOLDER
    return result


cleared = clear_old_tool_results(long_history, keep_last=1)
print(check_history(cleared))
print("估算 token:", estimate_tokens(long_history), "→", estimate_tokens(cleared))
for m in cleared[2:]:
    text = m.get("content") or "调用 " + m["tool_calls"][0]["function"]["arguments"]
    print(f"{m['role']:9s} {text[:60]}")

它的问题和截断类似，只是轻一些：两份日志里的关键数字随着工具结果一起被清除了。除非模型在之前的回复里**主动写下**了结论，否则这些信息就丢了。

那么，能不能在丢弃之前，先把有用的信息提炼出来？

## 方法三：摘要压缩（Compaction）

思路是：**让模型自己写一份工作摘要**，然后用这份摘要替换掉整段历史。

```text
压缩前：[system] [任务] [调用] [结果 3000 token] [调用] [结果 3000 token] ...
                      │
                      ▼  额外调用一次模型：「把上面的工作总结一下」
压缩后：[system] [任务 + 工作摘要 300 token]
```

摘要的质量取决于压缩提示词。好的压缩提示词要明确说出**必须保留什么**：原始任务、已确认的事实（文件名、数值、报错原文）、做过的操作、还没完成的事项。

In [ ]:
COMPACT_PROMPT = (
    "请把上面的对话压缩成一份工作摘要，供你自己之后继续工作使用。必须保留：\n"
    "1. 已经确认的事实和数据，文件名、数值、报错信息要写原文；\n"
    "2. 已经做过的操作和得出的结论；\n"
    "3. 还没完成的事项。\n"
    "只写摘要本身，不要编造没有出现过的信息。"
)


def compact(messages, offline_script):
    task = messages[1]["content"]  # 原始任务：原文保留，不交给模型概括
    request = messages + [{"role": "user", "content": COMPACT_PROMPT}]
    reply = ma.call_model(request, None, offline_script)  # 压缩时不提供工具
    summary = reply["choices"][0]["message"]["content"]
    return [
        messages[0],
        {"role": "user", "content": (
            f"原始任务：{task}\n\n"
            f"【上下文已压缩】以下是你之前工作的摘要：\n{summary}\n\n"
            "请在此基础上继续完成任务。"
        )},
    ]


compacted = compact(long_history, ["compact_summary_slow.json"])
print(check_history(compacted))
print("估算 token:", estimate_tokens(long_history), "→", estimate_tokens(compacted))
print()
print(compacted[1]["content"])

体积缩小到原来的一小部分，关键信息也都还在：两周的 `num_workers`、`data_time`，以及「变慢来自数据加载」的结论。

注意 `compact` 里的两个设计：

- **原始任务原文保留**，不让模型概括。任务描述里的每个字都可能是约束（「不要改 tests 目录」），概括时很容易丢
- **压缩时不提供工具**，让模型专心写摘要，不会顺手又去调用工具

Claude Code 的 `/compact` 命令和上下文快满时的自动压缩，原理就是这样。很多 Agent 工具都提供类似的功能。

### 摘要压缩的代价

摘要压缩是最「聪明」的方法，但它并不免费，也不可靠：

| 代价 | 说明 |
| --- | --- |
| **有损** | 摘要里没写的细节就永远丢了。哪些细节以后有用，压缩时往往判断不准 |
| **可能出错** | 摘要是模型写的，它可能写错数字，甚至「总结」出没发生过的事 |
| **要额外花钱** | 压缩本身要把整段长历史发送一遍，再生成摘要 |
| **打断缓存** | 历史被整个替换，之前的缓存失效（见下文的 prompt caching） |

所以实际使用 Agent 时，压缩之后最好检查一下：它还记得你最关键的那几条要求吗？

## 给 mini agent 加上自动压缩

现在把压缩放进 02 节的循环里。和 02 节的 `run_agent` 相比，只多了**调用模型之前的三行**：估算长度，超过阈值就先压缩。

为了在这个小任务里就能看到效果，阈值设得很小。真实的 Agent 通常在接近上下文窗口上限（例如用掉 80% 左右）时才压缩。

In [ ]:
COMPACT_THRESHOLD = 250  # 估算 token 超过这个数就压缩（为了演示故意设得很小）


def run_agent_with_compaction(task, offline_script=None, max_steps=10):
    offline_script = list(offline_script or [])
    messages = [
        {"role": "system", "content": ma.SYSTEM_PROMPT},
        {"role": "user", "content": task},
    ]
    for step in range(1, max_steps + 1):
        size = estimate_tokens(messages)                  # ← 新增
        if size > COMPACT_THRESHOLD:                      # ← 新增
            print(f"📦 估算 {size} token，超过阈值，开始压缩")
            messages = compact(messages, offline_script)  # ← 新增
            print(f"📦 压缩后估算 {estimate_tokens(messages)} token")

        reply = ma.call_model(messages, ma.TOOL_SPECS, offline_script)
        choice = reply["choices"][0]
        message = choice["message"]
        messages.append(message)
        ma.print_step(step, reply, message)

        tool_calls = message.get("tool_calls") or []
        if not tool_calls:
            return message.get("content"), messages
        ma.run_tool_calls(tool_calls, messages)

    print(f"⚠️ 达到最大步数 {max_steps}，强制停止")
    return None, messages

In [ ]:
ma.setup_workspace()
answer, history = run_agent_with_compaction(
    "工作区里的训练日志中，最低的 loss 是多少？出现在第几个 epoch？",
    offline_script=[
        "agent_list_dir.json",       # 第 1 步：列目录
        "agent_read_log.json",       # 第 2 步：读日志
        "compact_summary_loss.json",  # 压缩：生成摘要
        "agent_loss_answer.json",    # 第 3 步：回答
    ],
)
print("\n最终回答:", answer)
print("\n最终的消息历史:")
for m in history:
    print(f"{m['role']:9s} {(m.get('content') or '').replace(chr(10), ' | ')[:80]}")

读完日志后，历史超过了阈值，Agent 先压缩，再基于摘要回答。最终的历史只剩下 3 条消息。这个例子里的日志很短，所以压缩前后的差别不大；历史里有几份大文件时，效果就会像前面 `long_history` 那样明显。

> 在线模式下，摘要由你配置的模型现场生成，内容会和离线版本不同。可以对比一下：它保留了哪些信息，丢掉了哪些？

## 方法四：把记忆写到上下文外面

前三种方法都是在上下文**内部**取舍。还有一种思路：把重要的信息写进**文件**，需要时再读回来。上下文怎么压缩，文件里的内容都不会丢。

真实的 Agent 工具大量使用这种方法：

| 做法 | 例子 |
| --- | --- |
| 项目说明文件 | `AGENTS.md`、`CLAUDE.md` 在每次会话开始时自动加入系统提示词。系统提示词不参与压缩，所以写在里面的规则一直有效（第 07 节讲） |
| 任务清单 | Agent 把计划写成待办列表，每完成一项就勾掉，压缩后看清单就知道做到哪了 |
| 进度笔记 | 长任务中让 Agent 定期把「已确认的结论、下一步计划」写进一个笔记文件 |
| 子 Agent | 把「读 50 个文件找出某个函数」这类脏活交给另一个 Agent，它只把结论返回给主 Agent，主 Agent 的上下文里不会出现那 50 个文件 |

## Prompt Caching：让重复的前缀更便宜

Agent 每一步发送的历史，**开头部分几乎完全相同**：系统提示词、工具说明书、前面几轮对话。很多服务商因此提供了 **prompt caching（提示词缓存）**：如果这次请求的开头和最近某次请求一样，这部分就不必重新计算，费用也有折扣。

不同服务商的规则不同：有的自动生效，有的需要在请求里标记缓存位置，折扣力度和缓存有效时间也不一样，以官方文档为准。但有一条规律是共通的：**缓存按前缀匹配，前面的内容一变，后面就全部失效。**

这带来两个实践上的结论：

- 系统提示词、工具列表要尽量稳定，不要在每一步都改动它们
- 压缩会替换整段历史，压缩之后的第一次请求无法命中缓存。这也是 Agent 不会频繁压缩的原因之一

## 作为使用者，你能做什么

学完原理，再看日常使用 Agent 时的几条建议，就知道背后的原因了：

| 建议 | 原因 |
| --- | --- |
| 一个任务结束后，开始不相关的新任务时，**新开一个会话** | 旧任务的历史对新任务没用，只会占地方、干扰模型 |
| 长任务让 Agent **把计划和进度写进文件** | 压缩丢不掉文件里的内容 |
| 重要的规则写进 **`AGENTS.md` / `CLAUDE.md`**，而不是只在对话里说一次 | 对话内容可能被压缩掉，项目说明文件每次都会加载 |
| 压缩之后，**确认它还记得关键要求** | 摘要是有损的 |
| 不要让 Agent 一次性读入巨大的文件或日志，先让它用 `grep`、`head` 找到相关部分 | 第 2 章学的工具，正好用来给 Agent 节省上下文 |

## 动手试试

不计分。前四题在离线模式下也能完成。

1. `trim_history` 为什么要从 `assistant` 消息处开始保留，而不是从任意位置开始？如果从 `tool` 消息开始会怎样？
2. 把 `clear_old_tool_results` 的 `keep_last` 改成 `0` 和 `3`，分别对 `long_history` 运行，比较估算的 token 数。
3. 压缩后，用户追问「日志里哪种 ERROR 出现得最多？」。根据 `compact_summary_loss.json` 里的摘要，模型能直接回答吗？它应该怎么做？
4. 你让 Agent 做一个很长的重构任务，一开始就说了「不要修改 `tests/` 目录」。做到一半，它开始改测试文件了。结合本节内容，说出可能的原因和两种应对方法。
5. （在线）修改 `COMPACT_PROMPT`，要求摘要必须保留「每一种 ERROR 的名称和次数」，再运行一次自动压缩的例子，检查摘要是否做到了。

<details>
<summary>参考答案</summary>

**第 1 题**：一轮工具调用由 `assistant`（发起调用）和紧随其后的 `tool`（结果）组成，必须成对出现。从 `assistant` 处开始，保留下来的每一轮都是完整的；如果从 `tool` 消息开始，它找不到发起它的调用，API 会拒绝这个请求，就像「天真截断」那样。

**第 2 题**：结果可能和直觉相反。`keep_last=0` 反而比 `keep_last=1` 略大，因为最后一个结果（config.json）本来就比占位文字短，替换后变长了；`keep_last=3` 只清除了 `list_dir` 的短输出，两份大日志都还在，估算值甚至比原来还大一点。结论是：应该按**大小**决定清除哪些结果，而不是按条数。一个改进是只清除超过一定长度的工具结果。

**第 3 题**：不能。摘要里只写了「另有 3 条 ERROR」，没有写类型。模型应该重新调用 `read_file` 或 `run_command` 去查，而不是凭印象猜。这就是「有损」的具体体现：压缩时无法预知用户之后会问什么。

**第 4 题**：可能的原因是，这条要求只在对话开头说了一次，长任务中途发生了上下文压缩，摘要里没有保留它（或者它被淹没在大量工具输出里）。应对方法：

- 把这条规则写进项目的 `AGENTS.md` / `CLAUDE.md`，它每次都会加载，不受压缩影响
- 在对话中重新强调，并要求它撤销对测试文件的修改
- 更强制的做法是用 Hook 或权限设置，直接禁止写入 `tests/`（第 05、09 节讲）

**第 5 题**：开放题。重点是检查摘要里的数字和原日志是否一致：应该是 `dataloader_timeout` 2 次、`camera_frame_dropped` 1 次。

</details>

## 小结

| 你应该能 | 相关内容 |
| --- | --- |
| 说出 Agent 上下文变长的原因，以及贵、慢、超限、干扰四个问题 | 问题从哪里来 |
| 解释为什么截断要按「轮」进行，并保留原始任务 | 方法一 |
| 说出清除旧工具结果的做法和局限 | 方法二 |
| 写出摘要压缩的流程，说出它的四个代价 | 方法三 |
| 解释为什么写进文件的信息不会被压缩丢掉 | 方法四 |
| 说出 prompt caching 按前缀匹配的含义 | Prompt Caching |
| 在使用 Agent 时，用上下文的原理解释「新开会话」「写进 AGENTS.md」等建议 | 作为使用者 |

下一节：[04-MCP](./04-MCP.ipynb)